In [ ]:
def change_variable_name_and_merge_per_folder(in_paths, base_out_path, pattern,
                                              old_var_name, new_var_name,
                                              merged_file_name="wa_all_levels.nc",
                                              skip_existing=True,
                                              has_level=True,
                                              level_extract_func=None):
    """
    批量更改NetCDF文件中的变量名，添加level维度（如果需要），并在每个子文件夹生成合并文件
    同时保留输入子文件夹结构
    
    Parameters:
    -----------
    skip_existing : bool
        如果为True，跳过已存在的单个文件和合并文件（默认True）
    has_level : bool
        如果为True，处理3D数据（从文件名提取level维度）
        如果为False，处理2D数据（直接合并，不添加level维度）
    level_extract_func : callable, optional
        自定义level提取函数，输入为filename，输出为level值（int）
        如果为None，使用默认的智能提取方法
    """
    import os
    import glob
    import re
    import xarray as xr
    
    def default_extract_level(filename):
        """
        智能提取level值，支持多种文件名格式：
        - divergence_lev55.nc -> 55
        - ua_lev_031.nc -> 31
        - pfull_lev_90.nc -> 90
        - hus_lev_001.nc -> 1
        """
        # 方法1: 使用正则表达式匹配 lev 后面的数字
        match = re.search(r'lev[_-]?(\d+)', filename, re.IGNORECASE)
        if match:
            return int(match.group(1))
        
        # 方法2: 尝试从下划线分隔中查找数字
        parts = filename.replace('.nc', '').split('_')
        for part in parts:
            if part.isdigit():
                return int(part)
        
        raise ValueError(f"无法从文件名提取level: {filename}")

    for in_path in in_paths:
        # 当前输入文件夹名称
        folder_name = os.path.basename(os.path.normpath(in_path))
        out_path = os.path.join(base_out_path, folder_name)
        os.makedirs(out_path, exist_ok=True)
        
        # 检查合并文件是否已存在
        merged_file = os.path.join(out_path, merged_file_name)
        if skip_existing and os.path.exists(merged_file):
            print(f"✅ {folder_name}: 合并文件已存在，跳过处理")
            print(f"   文件: {merged_file}")
            continue

        file_pattern = os.path.join(in_path, pattern)
        input_files = sorted(glob.glob(file_pattern))
        
        if not input_files:
            print(f"⚠️ 文件夹 {folder_name} 没有匹配的文件 (pattern: {pattern})")
            continue
        
        data_type = "3D (多层级)" if has_level else "2D (单层/时间序列)"
        print(f"\n{'='*70}")
        print(f"🔄 处理文件夹: {folder_name} [{data_type}]")
        print(f"   输入路径: {in_path}")
        print(f"   输出路径: {out_path}")
        print(f"   找到文件数: {len(input_files)}")
        print(f"{'='*70}")
        
        # 选择level提取函数
        extract_level = level_extract_func if level_extract_func else default_extract_level
        
        datasets = []
        processed_count = 0

        for file in input_files:
            filename = os.path.basename(file)

            # 直接处理文件，不保存单个文件
            try:
                if has_level:
                    # 3D数据：使用提取函数获取level
                    try:
                        level = extract_level(filename)
                    except Exception as e:
                        print(f"⚠️ 无法从文件名提取 level: {filename}, 错误: {str(e)}, 跳过")
                        continue
                
                with xr.open_dataset(file) as ds:
                    # 修改变量名（如果需要）
                    if old_var_name in ds and old_var_name != new_var_name:
                        ds = ds.rename({old_var_name: new_var_name})
                    
                    # 3D数据需要添加 level 维度
                    if has_level:
                        ds = ds.expand_dims({"level": [level]})
                    
                    processed_count += 1
                    
                    if processed_count % 5 == 0 or processed_count == len(input_files):
                        print(f"✅ [{processed_count}/{len(input_files)}] 已处理: {filename}")
                    
                    datasets.append(ds)
            except Exception as e:
                print(f"❌ 处理失败: {filename} - {str(e)}")
                continue

        # 统计信息
        print(f"\n📊 处理统计:")
        print(f"   已处理: {processed_count} 个文件")
        print(f"   总计: {len(datasets)} 个文件用于合并")

        # 每个子文件夹单独合并
        if datasets:
            try:
                if has_level:
                    # 3D数据：沿level维度合并
                    ds_all = xr.concat(datasets, dim="level")
                else:
                    # 2D数据：直接合并（沿时间或其他维度）
                    ds_all = xr.concat(datasets, dim="time") if "time" in datasets[0].dims else xr.merge(datasets)
                
                var_data = ds_all[new_var_name]
                var_data.to_netcdf(merged_file)
                print(f"🎉 {folder_name} 合并完成!")
                print(f"   保存到: {merged_file}")
                print(f"   形状: {var_data.shape}")
                print(f"   维度: {list(var_data.dims)}")
            except Exception as e:
                print(f"❌ 合并失败: {str(e)}")
        else:
            print(f"⚠️ 文件夹 {folder_name} 没有可处理的文件！")


: 

In [ ]:

import os

input_folders_pfull_3d = [
    "/work/mh1498/m301257/processed_data/3d_layers/pfull_cntl_layers",
    "/work/mh1498/m301257/processed_data/3d_layers/pfull_p4k_layers",
    "/work/mh1498/m301257/processed_data/3d_layers/pfull_4co2_layers",

]

input_folders_phalf_3d = [
    "/work/mh1498/m301257/data_origin/phalf_cntl_layers/",
    "/work/mh1498/m301257/data_origin/phalf_p4k_layers/",
    "/work/mh1498/m301257/data_origin/phalf_4co2_layers/",

]

input_folders_ua_half_3d= [
    "/work/mh1498/m301257/converted_data/ua_3d/cntl/",
    "/work/mh1498/m301257/converted_data/ua_3d/p4k/",
    "/work/mh1498/m301257/converted_data/ua_3d/4co2",
]

input_folders_va_half_3d= [
    "/work/mh1498/m301257/converted_data/va_3d/cntl/",
    "/work/mh1498/m301257/converted_data/va_3d/p4k/",
    "/work/mh1498/m301257/converted_data/va_3d/4co2",
]

input_folders_rho_half_3d =[
    '/work/mh1498/m301257/processed_data/3d_layers/rho_cntl_layers',
    '/work/mh1498/m301257/processed_data/3d_layers/rho_p4k_layers',
    '/work/mh1498/m301257/processed_data/3d_layers/rho_4co2_layers'
    
    
]

input_folders_div_3d =[
    '/work/mh1498/m301257/3D_data/divergence_2d/cntl',
    '/work/mh1498/m301257/3D_data/divergence_2d/p4k',
    '/work/mh1498/m301257/3D_data/divergence_2d/4co2'
]

# # 输出路径（合并后的数据）
base_output_folder = "/work/mh1498/m301257/3D_data"
os.makedirs(base_output_folder, exist_ok=True)
    
print("="*70)
print("📦 开始合并处理后的数据")
print("="*70)
print(f"输出目录: {base_output_folder}")
change_variable_name_and_merge_per_folder(
    in_paths=input_folders_div_3d,
    base_out_path=base_output_folder,
    pattern="divergence_lev*_*.nc",
    old_var_name="divergence",
    new_var_name="divergence",
    merged_file_name="divergence_all_levels.nc",
    skip_existing=True,
    has_level=True
)
# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders_rho_half_3d,
#     base_out_path=base_output_folder,
#     pattern="rho_lev_*.nc",
#     old_var_name="rho",
#     new_var_name="rho",
#     merged_file_name="rho_all_levels.nc",
#     skip_existing=True,
#     has_level=True  # 3D数据，有level维度
# )
# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders_pfull_3d,
#     base_out_path=base_output_folder,
#     pattern="pfull_lev_*.nc",
#     old_var_name="pfull",
#     new_var_name="pfull",
#     merged_file_name="pfull_all_levels.nc",
#     skip_existing=True,
#     has_level=True  # 3D数据，有level维度
# )

# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders_phalf_3d,
#     base_out_path=base_output_folder,
#     pattern="phalf_lev_*.nc",
#     old_var_name="phalf",
#     new_var_name="phalf",
#     merged_file_name="phalf_all_levels.nc",
#     skip_existing=True,
#     has_level=True  # 3D数据，有level维度
# )

# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders_ua_half_3d,
#     base_out_path=base_output_folder,
#     pattern="ua_lev_*.nc",
#     old_var_name="ua",
#     new_var_name="ua",
#     merged_file_name="ua_all_levels.nc",
#     skip_existing=True,
#     has_level=True  # 3D数据，有level维度
# )
# change_variable_name_and_merge_per_folder(
#     in_paths=input_folders_va_half_3d,
#     base_out_path=base_output_folder,
#     pattern="va_lev_*.nc",
#     old_var_name="va",
#     new_var_name="va",
#     merged_file_name="va_all_levels.nc",
#     skip_existing=True,
#     has_level=True  # 3D数据，有level维度
# )
print("\n" + "="*70)
print("✅ 所有数据合并完成!")
print(f"合并后数据保存在: {base_output_folder}")
print("="*70)


📦 开始合并处理后的数据
输出目录: /work/mh1498/m301257/3D_data

🔄 处理文件夹: cntl [3D (多层级)]
   输入路径: /work/mh1498/m301257/3D_data/divergence_2d/cntl
   输出路径: /work/mh1498/m301257/3D_data/cntl
   找到文件数: 17
✅ [5/17] 已处理: divergence_lev58_cntl.nc
✅ [5/17] 已处理: divergence_lev58_cntl.nc
✅ [10/17] 已处理: divergence_lev76_cntl.nc
✅ [10/17] 已处理: divergence_lev76_cntl.nc
✅ [15/17] 已处理: divergence_lev84_cntl.nc
✅ [15/17] 已处理: divergence_lev84_cntl.nc
✅ [17/17] 已处理: divergence_lev87_cntl.nc

📊 处理统计:
   已处理: 17 个文件
   总计: 17 个文件用于合并
✅ [17/17] 已处理: divergence_lev87_cntl.nc

📊 处理统计:
   已处理: 17 个文件
   总计: 17 个文件用于合并


In [ ]:
# 🧪 测试level提取功能 - 验证支持多种文件名格式
import re

def test_extract_level():
    """测试不同文件名格式的level提取"""
    test_cases = [
        "divergence_lev55_cntl.nc",
        "ua_lev_031.nc",
        "pfull_lev_90.nc",
        "hus_lev_001.nc",
        "wa_lev41.nc",
        "divergence_lev78_p4k.nc"
    ]
    
    def default_extract_level(filename):
        # 方法1: 使用正则表达式匹配 lev 后面的数字
        match = re.search(r'lev[_-]?(\d+)', filename, re.IGNORECASE)
        if match:
            return int(match.group(1))
        
        # 方法2: 尝试从下划线分隔中查找数字
        parts = filename.replace('.nc', '').split('_')
        for part in parts:
            if part.isdigit():
                return int(part)
        
        raise ValueError(f"无法从文件名提取level: {filename}")
    
    print("="*60)
    print("🧪 测试level提取功能")
    print("="*60)
    
    for filename in test_cases:
        try:
            level = default_extract_level(filename)
            print(f"✅ {filename:35s} → level={level}")
        except Exception as e:
            print(f"❌ {filename:35s} → 错误: {str(e)}")
    
    print("="*60)

# 运行测试
test_extract_level()

🧪 测试level提取功能
✅ divergence_lev55_cntl.nc            → level=55
✅ ua_lev_031.nc                       → level=31
✅ pfull_lev_90.nc                     → level=90
✅ hus_lev_001.nc                      → level=1
✅ wa_lev41.nc                         → level=41
✅ divergence_lev78_p4k.nc             → level=78
